# Random Forests & Extra Trees
Accompanies **Chapter 13: Decision Trees** and **Chapter 14: Bagging, Random Forests and Extra Trees**.

The variance of an ensemble of $B$ trees with individual variance $\sigma^2$ and pairwise correlation $\rho$ is:
$$\text{Var}(\bar{f}) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$
As $B \to \infty$, the second term vanishes, but the first term $\rho \sigma^2$ remains. 

To drive total variance lower, we must reduce tree correlation $\rho$:
1. **Bootstrap Aggregation (Bagging):** Each tree trains on a random bootstrap sample with replacement.
2. **Feature Subsampling ($m = \sqrt{p}$):** Each split evaluates only a random subset of features, preventing dominant features from dictating every tree's root.
3. **Out-Of-Bag (OOB) Estimation:** Each bootstrap sample omits $\lim_{N\to\infty} (1 - 1/N)^N = 1/e \approx 36.8\%$ of observations, providing a free validation set.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

data_path = os.path.abspath('../../data/diabetic_data.csv')
df = pd.read_csv(data_path)

numeric_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
X = df[numeric_cols].fillna(0)
y = (df['readmitted'] == '<30').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Tracking Out-Of-Bag (OOB) Stabilization
We evaluate how OOB accuracy stabilizes as ensemble size $B$ grows.

In [ ]:
tree_counts = [10, 20, 40, 80, 120, 160]
oob_scores = []

for b in tree_counts:
    rf = RandomForestClassifier(n_estimators=b, max_features='sqrt', oob_score=True, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    oob_scores.append(rf.oob_score_)
    print(f"Estimators: {b:3d} | OOB Score: {rf.oob_score_:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(tree_counts, oob_scores, marker='o', color='navy', linewidth=2)
plt.xlabel('Number of Trees ($B$)', fontsize=11)
plt.ylabel('OOB Accuracy', fontsize=11)
plt.title('Out-Of-Bag Error Stabilization vs. Ensemble Size', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()